# 教材插图人物统计分析
汇总多本教材的分类结果，进行性别×职业交叉统计。

支持两种统计维度：
- **人头计数**：每个识别到的人物计一次
- **独立职业**：每页每种职业只计一次（去重后）

支持两种分类粒度：
- **一位码**（0-10，11类）
- **三位码**（10100-70400，78类 + 非从业人员0/9/10）

In [ ]:
import pandas as pd
from pathlib import Path
from collections import defaultdict

PROJECT_ROOT = Path("..").resolve()
RESULT_DIR = PROJECT_ROOT / "结果"
RECOGNITION_DIR = RESULT_DIR / "1.识别结果"
CLASS_1_DIR = RESULT_DIR / "2.分类结果" / "一位码"
CLASS_3_DIR = RESULT_DIR / "2.分类结果" / "三位码"
OUTPUT_DIR = RESULT_DIR / "3.统计结果"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"识别结果: {RECOGNITION_DIR}")
print(f"一位码分类: {CLASS_1_DIR}")
print(f"三位码分类: {CLASS_3_DIR}")
print(f"统计输出: {OUTPUT_DIR}")

## 1. 加载并汇总所有分类结果

In [ ]:
def load_all_from_dir(directory: Path) -> pd.DataFrame:
    """加载目录下所有Excel文件并合并，添加来源文件名列"""
    frames = []
    files = list(directory.glob("*.xlsx"))
    for f in files:
        df = pd.read_excel(f)
        df["来源文件"] = f.stem
        frames.append(df)
        print(f"  加载: {f.name} ({len(df)} 行)")
    if frames:
        return pd.concat(frames, ignore_index=True)
    return pd.DataFrame()

# 加载一位码分类结果
print("=== 一位码分类结果 ===")
df_class1_all = load_all_from_dir(CLASS_1_DIR)

print(f"\n=== 三位码分类结果 ===")
df_class3_all = load_all_from_dir(CLASS_3_DIR)

print(f"\n汇总: 一位码 {len(df_class1_all)} 行, 三位码 {len(df_class3_all)} 行")

## 2. 统计函数定义

In [ ]:
def gender_summary(df: pd.DataFrame, label: str = "") -> pd.DataFrame:
    """性别总体分布"""
    if df.empty:
        return pd.DataFrame()
    counts = df["gender"].value_counts()
    result = pd.DataFrame({"数量": counts, "占比": (counts / counts.sum() * 100).round(1)})
    if label:
        print(f"\n--- {label}: 性别分布 ---")
        print(result.to_string())
    return result

def gender_x_profession(df: pd.DataFrame, code_col: str = "职业分类代码",
                        name_col: str = "职业分类名称") -> pd.DataFrame:
    """性别×职业分类交叉表"""
    if df.empty or code_col not in df.columns:
        return pd.DataFrame()
    ct = pd.crosstab(
        [df[code_col], df[name_col]],
        df["gender"],
        margins=True, margins_name="合计"
    )
    return ct

def profession_ranking(df: pd.DataFrame, code_col: str = "职业分类代码",
                       name_col: str = "职业分类名称", top_n: int = 20) -> pd.DataFrame:
    """职业分类频次排名"""
    if df.empty or code_col not in df.columns:
        return pd.DataFrame()
    counts = df.groupby([code_col, name_col]).size().reset_index(name="数量")
    counts = counts.sort_values("数量", ascending=False).head(top_n)
    return counts

def gender_ratio_by_profession(df: pd.DataFrame, code_col: str = "职业分类代码",
                               name_col: str = "职业分类名称") -> pd.DataFrame:
    """每个职业分类的男女比例"""
    if df.empty or code_col not in df.columns:
        return pd.DataFrame()
    ct = pd.crosstab([df[code_col], df[name_col]], df["gender"])
    for col in ["男", "女", "未知"]:
        if col not in ct.columns:
            ct[col] = 0
    ct["合计"] = ct.sum(axis=1)
    ct["男占比%"] = (ct["男"] / ct["合计"] * 100).round(1)
    ct["女占比%"] = (ct["女"] / ct["合计"] * 100).round(1)
    return ct.sort_values("合计", ascending=False)

print("统计函数定义完成")

## 3. 执行统计（一位码）

In [ ]:
if not df_class1_all.empty:
    # 区分人头计数表和独立职业表
    mask_head = df_class1_all["来源文件"].str.contains("人头计数")
    mask_unique = df_class1_all["来源文件"].str.contains("独立职业")

    df_1_head = df_class1_all[mask_head].copy()
    df_1_unique = df_class1_all[mask_unique].copy()
    # 如果没有明确区分，全部当作人头计数
    if df_1_head.empty and df_1_unique.empty:
        df_1_head = df_class1_all.copy()

    print(f"一位码 - 人头计数: {len(df_1_head)} 行, 独立职业: {len(df_1_unique)} 行")

    # 性别分布
    gender_summary(df_1_head, "一位码-人头计数")
    if not df_1_unique.empty:
        gender_summary(df_1_unique, "一位码-独立职业")

    # 性别×职业交叉表
    print("\n=== 一位码 性别×职业 交叉表（人头计数）===")
    ct1_head = gender_x_profession(df_1_head)
    display(ct1_head)

    # 职业排名
    print("\n=== 一位码 职业频次排名（人头计数）===")
    display(profession_ranking(df_1_head))

    # 各职业男女比例
    print("\n=== 一位码 各职业男女比例（人头计数）===")
    display(gender_ratio_by_profession(df_1_head))
else:
    print("暂无一位码分类结果数据")

## 4. 执行统计（三位码）

In [ ]:
if not df_class3_all.empty:
    mask_head = df_class3_all["来源文件"].str.contains("人头计数")
    mask_unique = df_class3_all["来源文件"].str.contains("独立职业")

    df_3_head = df_class3_all[mask_head].copy()
    df_3_unique = df_class3_all[mask_unique].copy()
    if df_3_head.empty and df_3_unique.empty:
        df_3_head = df_class3_all.copy()

    print(f"三位码 - 人头计数: {len(df_3_head)} 行, 独立职业: {len(df_3_unique)} 行")

    gender_summary(df_3_head, "三位码-人头计数")

    print("\n=== 三位码 性别×职业 交叉表（人头计数，TOP20）===")
    ct3 = gender_x_profession(df_3_head)
    display(ct3)

    print("\n=== 三位码 职业频次排名 TOP20（人头计数）===")
    display(profession_ranking(df_3_head, top_n=20))

    print("\n=== 三位码 各职业男女比例（人头计数）===")
    display(gender_ratio_by_profession(df_3_head))
else:
    print("暂无三位码分类结果数据")

## 5. 导出统计结果到Excel

In [ ]:
output_file = OUTPUT_DIR / "统计汇总.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    # 一位码统计
    if not df_class1_all.empty:
        if not df_1_head.empty:
            gender_x_profession(df_1_head).to_excel(writer, sheet_name="一位码_性别×职业_人头")
            gender_ratio_by_profession(df_1_head).to_excel(writer, sheet_name="一位码_男女比例_人头")
            profession_ranking(df_1_head).to_excel(writer, sheet_name="一位码_职业排名_人头", index=False)
        if not df_1_unique.empty:
            gender_x_profession(df_1_unique).to_excel(writer, sheet_name="一位码_性别×职业_独立")
            gender_ratio_by_profession(df_1_unique).to_excel(writer, sheet_name="一位码_男女比例_独立")

    # 三位码统计
    if not df_class3_all.empty:
        if not df_3_head.empty:
            gender_x_profession(df_3_head).to_excel(writer, sheet_name="三位码_性别×职业_人头")
            gender_ratio_by_profession(df_3_head).to_excel(writer, sheet_name="三位码_男女比例_人头")
            profession_ranking(df_3_head, top_n=50).to_excel(writer, sheet_name="三位码_职业排名_人头", index=False)
        if not df_3_unique.empty:
            gender_x_profession(df_3_unique).to_excel(writer, sheet_name="三位码_性别×职业_独立")
            gender_ratio_by_profession(df_3_unique).to_excel(writer, sheet_name="三位码_男女比例_独立")

print(f"统计汇总已导出: {output_file}")